# Exportar datos para un dashboard Xila

Plantilla del final de cada notebook de limpieza. La limpieza pesada va arriba, en pandas; aquí solo se valida el contrato (`docs/contrato-csv.md`) y se escribe el CSV en formato largo.

Si algo no cumple el contrato, `exportar_xila` lanza `ContratoError` con la columna y ejemplos de los valores que fallan, **antes** de escribir el archivo.

In [ ]:
import sys
sys.path.append('../python')  # o copia xila_export.py junto al notebook

import pandas as pd
from xila_export import exportar_xila, ContratoError

## 1. Datos limpios en formato largo

Una fila por observación, una columna por variable. Si tienes una columna por región (formato ancho), `df.melt(...)` lo convierte.

In [ ]:
df = pd.DataFrame({
    'Fecha': pd.date_range('2026-01-01', periods=6, freq='D').repeat(2),
    'Región': ['Norte', 'Sur'] * 6,
    'Monto (MXN)': [1250.5, 980, 1100, None, 1320, 1010, 1180, 990, 1400, 1020, 1290, 975],
})
df.head()

## 2. Exportar

`exportar_xila(df, 'ventas')` pasa los nombres a `snake_case` (`monto_mxn`), las fechas a ISO 8601 en UTC y los nulos a celda vacía; escribe `data/ventas.csv`, listo para `dashboard.json`.

In [ ]:
ruta = exportar_xila(df, 'ventas', carpeta='../clientes/mi-cliente/public/data')
print(ruta.read_text()[:200])

## 3. Qué pasa con un error

Una fecha mal formada o una columna que mezcla números con `'N/A'` se detiene aquí, en tu máquina.

In [ ]:
malo = df.copy()
malo['Monto (MXN)'] = malo['Monto (MXN)'].astype(object)
malo.loc[3, 'Monto (MXN)'] = 'N/A'
try:
    exportar_xila(malo, 'ventas', carpeta='/tmp')
except ContratoError as e:
    print('ContratoError:', e)